In [1]:
import torch
import torch.nn as nn
import time
import numpy as np
import os
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

In [2]:
DATA_ROOT = "ip102"

TRAIN_TXT = os.path.join(DATA_ROOT, "train.txt")
VAL_TXT   = os.path.join(DATA_ROOT, "val.txt")

TRAIN_DIR = os.path.join(DATA_ROOT, "classification/train")
VAL_DIR   = os.path.join(DATA_ROOT, "classification/val")

NUM_CLASSES = 102
BATCH_SIZE = 32
EPOCHS = 5
WORKERS = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter("runs/mob_mobilenetv2")

print("Device:", device)
print("Train TXT:", TRAIN_TXT)
print("Val TXT:", VAL_TXT)
print("Train DIR:", TRAIN_DIR)
print("Val DIR:", VAL_DIR)

Device: cuda
Train TXT: ip102\train.txt
Val TXT: ip102\val.txt
Train DIR: ip102\classification/train
Val DIR: ip102\classification/val


In [3]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [4]:
class IP102Dataset(Dataset):

    def __init__(self, txt_file, img_root, transform=None):

        print("\nLoading dataset:", txt_file)

        self.samples = []
        self.img_root = img_root
        self.transform = transform

        with open(txt_file) as f:
            lines = f.readlines()

        print("Lines found:", len(lines))

        for i, line in enumerate(lines):

            img_name, label = line.strip().split()
            label = int(label)

            img_path = os.path.join(
                img_root,
                str(label),
                img_name
            )

            if i < 5:
                print("Example path:", img_path)
                print("Exists:", os.path.exists(img_path))

            self.samples.append((img_path, label))

        print("Dataset loaded:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        if not os.path.exists(img_path):
            print("Missing file:", img_path)
            raise FileNotFoundError(img_path)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
train_ds = IP102Dataset(TRAIN_TXT, TRAIN_DIR, transform)
val_ds   = IP102Dataset(VAL_TXT, VAL_DIR, transform)

print("\nTrain samples:", len(train_ds))
print("Val samples:", len(val_ds))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=True
)

print("DataLoaders created")


Loading dataset: ip102\train.txt
Lines found: 45095
Example path: ip102\classification/train\0\00002.jpg
Exists: True
Example path: ip102\classification/train\0\00003.jpg
Exists: True
Example path: ip102\classification/train\0\00005.jpg
Exists: True
Example path: ip102\classification/train\0\00006.jpg
Exists: True
Example path: ip102\classification/train\0\00008.jpg
Exists: True
Dataset loaded: 45095

Loading dataset: ip102\val.txt
Lines found: 7508
Example path: ip102\classification/val\0\00009.jpg
Exists: True
Example path: ip102\classification/val\0\00012.jpg
Exists: True
Example path: ip102\classification/val\0\00014.jpg
Exists: True
Example path: ip102\classification/val\0\00034.jpg
Exists: True
Example path: ip102\classification/val\0\00035.jpg
Exists: True
Dataset loaded: 7508

Train samples: 45095
Val samples: 7508
DataLoaders created


In [6]:
model = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.DEFAULT
)

model.classifier[1] = nn.Linear(
    model.last_channel,
    NUM_CLASSES
)

model = model.to(device)

In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=7,
    gamma=0.1
)

In [8]:
for epoch in range(EPOCHS):

    model.train()
    running_loss = 0
    t0 = time.time()

    for batch_idx, (imgs, labels) in enumerate(train_loader):

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1} Batch {batch_idx}")

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"loss={running_loss:.4f} | "
        f"time={time.time()-t0:.1f}s"
    )
    writer.add_scalar("Loss/train", running_loss, epoch)

Epoch 1 Batch 0
Epoch 1 Batch 50
Epoch 1 Batch 100
Epoch 1 Batch 150
Epoch 1 Batch 200
Epoch 1 Batch 250
Epoch 1 Batch 300
Epoch 1 Batch 350
Epoch 1 Batch 400
Epoch 1 Batch 450
Epoch 1 Batch 500
Epoch 1 Batch 550
Epoch 1 Batch 600
Epoch 1 Batch 650
Epoch 1 Batch 700
Epoch 1 Batch 750
Epoch 1 Batch 800
Epoch 1 Batch 850
Epoch 1 Batch 900
Epoch 1 Batch 950
Epoch 1 Batch 1000
Epoch 1 Batch 1050
Epoch 1 Batch 1100
Epoch 1 Batch 1150
Epoch 1 Batch 1200
Epoch 1 Batch 1250
Epoch 1 Batch 1300
Epoch 1 Batch 1350
Epoch 1 Batch 1400
Epoch 1/5 | loss=3166.3108 | time=629.5s
Epoch 2 Batch 0
Epoch 2 Batch 50
Epoch 2 Batch 100
Epoch 2 Batch 150
Epoch 2 Batch 200
Epoch 2 Batch 250
Epoch 2 Batch 300
Epoch 2 Batch 350
Epoch 2 Batch 400
Epoch 2 Batch 450
Epoch 2 Batch 500
Epoch 2 Batch 550
Epoch 2 Batch 600
Epoch 2 Batch 650
Epoch 2 Batch 700
Epoch 2 Batch 750
Epoch 2 Batch 800
Epoch 2 Batch 850
Epoch 2 Batch 900
Epoch 2 Batch 950
Epoch 2 Batch 1000
Epoch 2 Batch 1050
Epoch 2 Batch 1100
Epoch 2 Batch 115

In [9]:
def evaluate(model, loader):

    model.eval()

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for i, (imgs, labels) in enumerate(loader):

            if i % 20 == 0:
                print("Eval batch:", i)

            imgs = imgs.to(device)

            outputs = model(imgs)

            preds = outputs.argmax(dim=1).cpu().numpy()

            preds_all.extend(preds)
            labels_all.extend(labels.numpy())

    acc = np.mean(
        np.array(preds_all) == np.array(labels_all)
    )

    return acc

In [10]:
val_acc = evaluate(model, val_loader)
writer.add_scalar("Accuracy/val", val_acc, 0)
writer.close()

print("Validation accuracy:", val_acc)

param_count = sum(p.numel() for p in model.parameters())

print("Parameters:", f"{param_count:,}")

Eval batch: 0
Eval batch: 20
Eval batch: 40
Eval batch: 60
Eval batch: 80
Eval batch: 100
Eval batch: 120
Eval batch: 140
Eval batch: 160
Eval batch: 180
Eval batch: 200
Eval batch: 220
Validation accuracy: 0.630793819925413
Parameters: 2,354,534


In [11]:
model.eval()

dummy = torch.randn(1,3,224,224).to(device)

for _ in range(10):
    model(dummy)

N = 100

t0 = time.time()

for _ in range(N):
    model(dummy)

avg = (time.time()-t0)/N

print(f"Inference time: {avg*1000:.2f} ms")

Inference time: 8.26 ms
